# Notebook 04: Real Quantization Benchmark -- FP16 vs. INT8/INT4 Memory & Latency

`[REAL]` Companion to Module 05. Real `bitsandbytes` quantization of `Qwen/Qwen2.5-0.5B-Instruct` on the RTX 4060 at FP16, INT8, and INT4 -- directly testing Module 05's central caveat: does a real memory reduction here translate into a proportional real latency reduction on this specific real hardware/kernel combination, or not? Whichever real outcome occurs is reported as-is.

In [1]:
import time
import statistics
import logging
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# bitsandbytes emits a real, repeated logging.warning() (not a Python warnings.warn(), which is
# why filterwarnings doesn't touch it) on every 8-bit matmul call; across many timed repeats this
# produces thousands of near-identical notebook output entries and bloats the saved file to
# double-digit MB for no informational gain -- suppressed via the logger itself, deliberately.
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Device: {DEVICE}")

D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 1. Real Memory Footprint: FP16 vs. INT8 vs. INT4

`[REAL]` Loading the same real model three times at three real precisions via `bitsandbytes`' `BitsAndBytesConfig`, reading each real model's `get_memory_footprint()`.

In [2]:
memory_results = {}

model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(DEVICE)
memory_results["FP16"] = model_fp16.get_memory_footprint() / 1024 / 1024
print(f"FP16 memory footprint: {memory_results['FP16']:.2f} MB")

bnb_int8_config = BitsAndBytesConfig(load_in_8bit=True)
model_int8 = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_int8_config, device_map=DEVICE)
memory_results["INT8"] = model_int8.get_memory_footprint() / 1024 / 1024
print(f"INT8 memory footprint: {memory_results['INT8']:.2f} MB")

bnb_int4_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model_int4 = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_int4_config, device_map=DEVICE)
memory_results["INT4"] = model_int4.get_memory_footprint() / 1024 / 1024
print(f"INT4 memory footprint: {memory_results['INT4']:.2f} MB")

print("\n(pending real interpretation)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  44%|████▍     | 128/290 [00:00<00:00, 1278.06it/s]

Loading weights:  88%|████████▊ | 256/290 [00:00<00:00, 1140.90it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1105.58it/s]

...

Loading weights:  74%|███████▍  | 215/290 [00:01<00:00, 251.92it/s]

Loading weights:  84%|████████▍ | 243/290 [00:01<00:00, 254.16it/s]

Loading weights:  93%|█████████▎| 269/290 [00:01<00:00, 248.55it/s]

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 209.77it/s]

INT4 memory footprint: 430.42 MB

(pending real interpretation)


**Real result:** `FP16` → `942.29 MB`, `INT8` → `601.04 MB` (`36.2%` real reduction), `INT4` → `430.42 MB` (`54.3%` real reduction). Both real memory reductions are genuine and substantial, exactly as Module 05's formula predicts — this half of the caveat holds cleanly. Section 2 tests whether that memory win comes with a proportional real speed win.

## 2. Real Generation Latency: FP16 vs. INT8 vs. INT4

`[REAL]` Measuring real generation latency for a fixed real prompt/output length across all three real precisions, using the same warm-up + repeated-run + `torch.cuda.synchronize()` + median/p95 methodology as Notebook 01.

In [3]:
def timed_generate(model, input_ids, n_new_tokens, n_repeats=6, n_warmup=1):
    def run():
        with torch.no_grad():
            model.generate(input_ids, max_new_tokens=n_new_tokens, min_new_tokens=n_new_tokens,
                           do_sample=False, pad_token_id=tokenizer.eos_token_id)

    for _ in range(n_warmup):
        run()
        torch.cuda.synchronize()

    samples = []
    for _ in range(n_repeats):
        torch.cuda.synchronize()
        start = time.perf_counter()
        run()
        torch.cuda.synchronize()
        samples.append(time.perf_counter() - start)
    return statistics.median(samples), sorted(samples)[int(0.95 * (len(samples) - 1))]

FIXED_PROMPT = "Explain the concept of gravity in simple terms."
N_NEW_TOKENS = 64
input_ids = tokenizer(FIXED_PROMPT, return_tensors="pt").input_ids.to(DEVICE)

latency_results = {}
for name, model in [("FP16", model_fp16), ("INT8", model_int8), ("INT4", model_int4)]:
    median_s, p95_s = timed_generate(model, input_ids, N_NEW_TOKENS)
    latency_results[name] = {"median_ms": median_s * 1000, "p95_ms": p95_s * 1000,
                              "tpot_median_ms": median_s * 1000 / N_NEW_TOKENS}
    print(f"{name}: median={median_s*1000:.2f}ms total, p95={p95_s*1000:.2f}ms, "
          f"TPOT={median_s*1000/N_NEW_TOKENS:.3f}ms/token")

print("\n(pending real interpretation)")

FP16: median=6208.57ms total, p95=6571.40ms, TPOT=97.009ms/token


INT8: median=20937.30ms total, p95=21487.78ms, TPOT=327.145ms/token


INT4: median=8325.03ms total, p95=8672.21ms, TPOT=130.079ms/token

(pending real interpretation)


## 3. Real Interpretation: Memory Won, Latency Lost — Reported As-Is

`[REAL]` The real result is the **opposite** of a proportional win, and is reported exactly as measured, per the signed-off plan's discipline: `FP16` was the *fastest* real configuration at `97.009ms/token`. `INT8` was `327.145ms/token` — a real **3.37x slowdown** versus FP16, despite its genuine `36.2%` memory reduction. `INT4` was `130.079ms/token` — a real **1.34x slowdown** versus FP16, despite its genuine `54.3%` memory reduction.

**This is a direct, real, measured confirmation of Module 05's central caveat**, in its strongest possible form: not just "less than proportional," but real quantization made generation slower on this specific hardware/kernel combination, not faster. The real, plausible explanation, consistent with Module 05's own framing: `bitsandbytes`' INT8/INT4 kernels perform real dequantization work on every matmul call, and at this small model size (0.5B params) and batch size (1) on a consumer GPU, that real per-call dequantization overhead outweighs the real memory-bandwidth savings — the workload here was never memory-bandwidth-bound enough (per Notebook 01's own real TPOT findings, which showed decode cost dominated by a largely fixed per-step floor) for the smaller real memory footprint to translate into a real speed win. `INT8` being slower than `INT4` here is itself a real, honest data point too — not a typo — plausibly reflecting `bitsandbytes`' specific INT8 kernel path being less optimized for this small-model/single-sequence regime than its INT4 path.

**Bottom line:** real memory reduction and real latency reduction are genuinely decoupled here, exactly as Module 05 warns — this notebook's own real numbers are a stronger, cleaner demonstration of that caveat than a "smaller win than expected" result would have been.